# Z24 Standard Time-Series Classification with tsai

**Author:** TRAN VAN PHUONG

This notebook converts the cleaned Z24 PDT data to the official tsai input contract and trains a PyTorch time-series classifier. It uses the same recording-level train, validation, and test isolation as the TensorFlow experiment.

## Time-series data contract

The tsai repository defines a three-dimensional input array with shape:

**samples x variables x sequence length**

For this experiment:

- one sample is one 10-second window;
- variables are the five common sensor channels;
- sequence length is 1,000 samples at 100 Hz;
- X therefore has shape (number of windows, 5, 1000);
- y has shape (number of windows,) and contains labels 0--16.

Reference: https://github.com/timeseriesAI/tsai

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import sys

start = Path.cwd().resolve()
ROOT = next(
    (path for path in (start, *start.parents)
     if (path / "src" / "data" / "pdt_training_data.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from the shm project or its notebooks folder")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from fastai.metrics import accuracy
from tsai.data.core import TSClassification
from tsai.data.validation import combine_split_data
from tsai.tslearner import TSClassifier

from src.data.pdt_training_data import (
    common_channels,
    latest_complete_run,
    load_windows,
    normalize_from_train,
    read_manifest,
    split_by_recording,
    to_tsai_format,
)

print("Project root:", ROOT)

## Configuration

ResNet is used as the first baseline because it is a standard time-series classifier included in tsai. AVT and FVT must be trained as separate experiments.

In [ ]:
MEASUREMENT = "avt"
WINDOW_SAMPLES = 1000
MODEL_ARCH = "ResNet"
BATCH_SIZE = 64
EPOCHS = 20
LEARNING_RATE = 1e-3
SEED = 42

## Read metadata and split recordings

The manifest is split before loading windows. Every window originating from one MAT recording remains entirely in train, validation, or test.

In [ ]:
RUN_DIR = latest_complete_run(ROOT, MEASUREMENT)
MANIFEST = read_manifest(RUN_DIR)
CHANNELS = list(common_channels(MANIFEST))
SPLITS = split_by_recording(MANIFEST, seed=SEED)
NUM_CLASSES = len({int(row["label"]) for row in MANIFEST})

split_summary = pd.DataFrame([
    {
        "split": name,
        "recordings": len({row["source"] for row in rows}),
        "csv_segments": len(rows),
        "expected_windows": sum(int(row["samples"]) // WINDOW_SAMPLES for row in rows),
    }
    for name, rows in SPLITS.items()
]).set_index("split")

print("Clean-data run:", RUN_DIR)
print("Channels:", CHANNELS)
print("Classes:", NUM_CLASSES)
display(split_summary)

assert CHANNELS == ["R1V", "R2L", "R2T", "R2V", "R3V"]
assert NUM_CLASSES == 17

## Create model-ready time-series arrays

load_windows first creates arrays in time-major format. Normalization uses training statistics only. to_tsai_format then moves the channel axis before the time axis.

In [ ]:
x_train, y_train = load_windows(RUN_DIR, SPLITS["train"], CHANNELS, WINDOW_SAMPLES)
x_validation, y_validation = load_windows(
    RUN_DIR, SPLITS["validation"], CHANNELS, WINDOW_SAMPLES
)
x_test, y_test = load_windows(RUN_DIR, SPLITS["test"], CHANNELS, WINDOW_SAMPLES)

x_train, x_validation, x_test, channel_mean, channel_std = normalize_from_train(
    x_train, x_validation, x_test
)

X_train = to_tsai_format(x_train)
X_validation = to_tsai_format(x_validation)
X_test = to_tsai_format(x_test)

del x_train, x_validation, x_test

In [ ]:
contract = []
for name, X_part, y_part in (
    ("train", X_train, y_train),
    ("validation", X_validation, y_validation),
    ("test", X_test, y_test),
):
    assert X_part.ndim == 3
    assert X_part.shape[1:] == (len(CHANNELS), WINDOW_SAMPLES)
    assert X_part.dtype == np.float32
    assert len(X_part) == len(y_part)
    assert np.isfinite(X_part).all()
    assert set(np.unique(y_part)) == set(range(NUM_CLASSES))
    contract.append({
        "split": name,
        "X shape": str(X_part.shape),
        "y shape": str(y_part.shape),
        "dtype": str(X_part.dtype),
        "labels": f"{y_part.min()}--{y_part.max()}",
    })

display(pd.DataFrame(contract).set_index("split"))
print("tsai time-series contract: OK")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
time_seconds = np.arange(WINDOW_SAMPLES) / 100.0
for channel_index, channel_name in enumerate(CHANNELS):
    axes[0].plot(time_seconds, X_train[0, channel_index], label=channel_name)
axes[0].set_title("One standardized time-series sample")
axes[0].set_xlabel("Time (seconds)")
axes[0].set_ylabel("Normalized value")
axes[0].legend()
axes[0].grid(alpha=0.3)

for name, labels in (
    ("train", y_train),
    ("validation", y_validation),
    ("test", y_test),
):
    axes[1].plot(
        range(1, NUM_CLASSES + 1),
        np.bincount(labels, minlength=NUM_CLASSES),
        marker="o",
        label=name,
    )
axes[1].set_title("Window count by condition")
axes[1].set_xlabel("Condition")
axes[1].set_ylabel("Windows")
axes[1].set_xticks(range(1, NUM_CLASSES + 1))
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Build the tsai classifier

Only train and validation are combined for TSClassifier. The predefined split indexes preserve their original roles. Test remains outside the learner until final evaluation.

In [ ]:
X, y, tsai_splits = combine_split_data(
    [X_train, X_validation],
    [y_train, y_validation],
)

timestamp = datetime.now().strftime("%d-%m-%Y_%H-%M-%S")
ARTIFACT_DIR = ROOT / "artifacts" / "tsai" / f"{MEASUREMENT}_{MODEL_ARCH}_{timestamp}"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)

learner = TSClassifier(
    X,
    y,
    splits=tsai_splits,
    tfms=[None, TSClassification()],
    arch=MODEL_ARCH,
    metrics=accuracy,
    bs=BATCH_SIZE,
    path=ARTIFACT_DIR,
    model_dir="models",
    seed=SEED,
    verbose=True,
)
learner.summary()

## Train

Validation is used during training. Test is not used by fit_one_cycle.

In [ ]:
learner.fit_one_cycle(EPOCHS, LEARNING_RATE)

## Final test evaluation

In [ ]:
probabilities, targets, predictions = learner.get_X_preds(X_test, y_test)
predictions = predictions.detach().cpu().numpy()

condition_names = [
    next(row["condition_name"] for row in MANIFEST if int(row["label"]) == label)
    for label in range(NUM_CLASSES)
]
report = classification_report(
    y_test,
    predictions,
    labels=np.arange(NUM_CLASSES),
    target_names=[f"{index + 1:02d}: {name}" for index, name in enumerate(condition_names)],
    output_dict=True,
    zero_division=0,
)
display(pd.DataFrame(report).T.round(3))

matrix = confusion_matrix(y_test, predictions, labels=np.arange(NUM_CLASSES))
fig, ax = plt.subplots(figsize=(12, 12))
ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=np.arange(1, NUM_CLASSES + 1),
).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"{MEASUREMENT.upper()} {MODEL_ARCH} test confusion matrix")
ax.set_xlabel("Predicted condition")
ax.set_ylabel("True condition")
plt.tight_layout()
plt.show()

## Save model and experiment metadata

In [ ]:
learner.export("z24_tsai_classifier.pkl")
pd.DataFrame(report).T.to_csv(ARTIFACT_DIR / "classification_report.csv")
np.savetxt(ARTIFACT_DIR / "confusion_matrix.csv", matrix, fmt="%d", delimiter=",")

experiment = {
    "measurement": MEASUREMENT,
    "model_architecture": MODEL_ARCH,
    "data_contract": "samples x variables x sequence_length",
    "train_shape": list(X_train.shape),
    "validation_shape": list(X_validation.shape),
    "test_shape": list(X_test.shape),
    "channels": CHANNELS,
    "sampling_rate_hz": 100,
    "window_samples": WINDOW_SAMPLES,
    "num_classes": NUM_CLASSES,
    "seed": SEED,
    "normalization_mean": channel_mean.tolist(),
    "normalization_std": channel_std.tolist(),
    "test_accuracy": float((predictions == y_test).mean()),
}
(ARTIFACT_DIR / "experiment.json").write_text(
    json.dumps(experiment, indent=2), encoding="utf-8"
)
print("Saved to:", ARTIFACT_DIR)